# World Cup 2026 - Task 2.2: dataset processing
Turns the 4 raw files into the final 208-row dataset (one row per team per match).
Inputs: the fbref schedule, the FIFA ranking, the historical results and the squad values.
The regression is in a separate notebook.

## 1. Setup

In [1]:
import re
from collections import defaultdict
from io import StringIO

import pandas as pd

# the four input files
SCHEDULE_FILE = "raw_fbref_schedule_export.csv"
RANKING_FILE = "raw_fifa_ranking_11june2026.csv"
HISTORY_FILE = "raw_international_results.csv"
SQUAD_FILE = "raw_squad_values_eur_m.csv"

## 2. Settings and lookups

In [2]:
KICKOFF = "2026-06-11"   # only matches before this date count as history
FORM_WINDOW = 10         # "recent form" = last 10 matches
FIRST_MATCH_REST = 7     # rest days given to every team before its first match

HOSTS = {"Mexico", "Canada", "United States"}

# the sources spell some team names differently, so everything is converted to one spelling
TEAM_NAME_FIX = {"Bosnia–Herz": "Bosnia and Herzegovina", "USA": "United States"}
HISTORY_NAME_FIX = {
    "Czech Republic": "Czechia", "Turkey": "Türkiye", "Iran": "IR Iran",
    "Cape Verde": "Cabo Verde", "Ivory Coast": "Côte d'Ivoire", "DR Congo": "Congo DR",
    "South Korea": "Korea Republic",
}

## 3. Match schedule (fbref)

In [3]:
def clean_fbref_team(raw):
    # fbref adds a small country code, e.g. "Mexico mx", which is removed here
    tokens = [t for t in str(raw).strip().split(" ") if not re.fullmatch(r"[a-z]{2,3}", t)]
    name = " ".join(tokens).strip()
    return TEAM_NAME_FIX.get(name, name)


def read_schedule(path):
    # reads the CSV, or the .xls that fbref exports (which is really an HTML table)
    if path.lower().endswith((".xls", ".html", ".htm")):
        with open(path, encoding="utf-8") as f:
            df = pd.read_html(StringIO(f.read()))[0]
    else:
        df = pd.read_csv(path, encoding="utf-8")
    return df.dropna(subset=["Round"])  # removes the blank separator rows


matches = []
for _, row in read_schedule(SCHEDULE_FILE).iterrows():
    score = str(row["Score"]).strip()
    core = re.sub(r"\(\d+\)", "", score).strip()  # removes the penalty shoot-out numbers
    goals1, goals2 = [int(x) for x in re.split(r"[–-]", core)]
    matches.append({
        "date": str(row["Date"]).strip(),
        "is_knockout": int(row["Round"].strip() != "Group stage"),
        "team1": clean_fbref_team(row["Home"]),
        "team2": clean_fbref_team(row["Away"]),
        "goals1": goals1,
        "goals2": goals2,
    })

print("Matches loaded:", len(matches))
assert len(matches) == 104

Matches loaded: 104


## 4. FIFA ranking points (11 June 2026) and squad values

In [4]:
rank_df = pd.read_csv(RANKING_FILE, encoding="utf-8")
rank_df["team"] = rank_df["team"].str.strip().replace(TEAM_NAME_FIX)
fifa_points = dict(zip(rank_df["team"], rank_df["points"].astype(float)))

squad_df = pd.read_csv(SQUAD_FILE, encoding="utf-8")
squad_df["team"] = squad_df["team"].str.strip().replace(TEAM_NAME_FIX)
squad_value = dict(zip(squad_df["team"], squad_df["squad_value_eur_m"].astype(float)))

teams = {m["team1"] for m in matches} | {m["team2"] for m in matches}
print("Teams:", len(teams))
print("Missing FIFA points:", [t for t in teams if t not in fifa_points])
print("Missing squad value:", [t for t in teams if t not in squad_value])

Teams: 48
Missing FIFA points: []
Missing squad value: []


## 5. History: attacking and defensive form
Only matches before the tournament are used, so no World Cup result leaks in.

In [5]:
hist = pd.read_csv(HISTORY_FILE, encoding="utf-8")
hist["home_team"] = hist["home_team"].replace(HISTORY_NAME_FIX)
hist["away_team"] = hist["away_team"].replace(HISTORY_NAME_FIX)
hist = hist[hist["date"] < KICKOFF]
hist = hist[hist["home_team"].isin(teams) | hist["away_team"].isin(teams)]
hist = hist.dropna(subset=["home_score", "away_score"])

team_games = defaultdict(list)   # team -> (date, goals for, goals against)
for d, h, a, hg, ag in zip(hist["date"], hist["home_team"], hist["away_team"],
                           hist["home_score"].astype(int), hist["away_score"].astype(int)):
    team_games[h].append((d, hg, ag))
    team_games[a].append((d, ag, hg))
for t in team_games:
    team_games[t].sort(key=lambda g: g[0])  # sorted oldest to newest


def last_games(team, before):
    return [g for g in team_games.get(team, []) if g[0] < before][-FORM_WINDOW:]


def goals_for_per_game(team, before):
    # how many goals the team scored per match in its last 10 matches
    games = last_games(team, before)
    return sum(gf for _, gf, _ in games) / len(games) if games else 0.0


def goals_against_per_game(team, before):
    # how many goals the team conceded per match in its last 10 matches
    games = last_games(team, before)
    return sum(ga for _, _, ga in games) / len(games) if games else 0.0


print("Teams with no history:", [t for t in teams if not last_games(t, KICKOFF)])

Teams with no history: []


## 6. Rest days
Days since the team's previous World Cup match. The kick-off date of every match is
known in advance, so this is pre-match information.

In [6]:
matches.sort(key=lambda m: m["date"])
last_played = {}
for m in matches:
    d = pd.Timestamp(m["date"])
    for side in ("team1", "team2"):
        t = m[side]
        m["rest_" + side] = (d - last_played[t]).days if t in last_played else FIRST_MATCH_REST
    last_played[m["team1"]] = d
    last_played[m["team2"]] = d

print("Rest days range:", min(min(m["rest_team1"], m["rest_team2"]) for m in matches),
      "to", max(max(m["rest_team1"], m["rest_team2"]) for m in matches))

Rest days range: 3 to 8


## 7. Build the dataset
Two rows per match, one from each team's point of view (teams in alphabetical order).
`row_id`, `match_id`, `date`, `team` and `opp` are just labels.

| Column | Meaning |
|---|---|
| `fifa_points_team` | the team's FIFA ranking points on 11 June 2026 |
| `squad_value_team_eur_m` | the team's squad market value (EUR millions) |
| `squad_value_opp_eur_m` | the opponent's squad market value (EUR millions) |
| `form_gf_pg_team` | goals the team scored per match in its last 10 matches |
| `form_ga_pg_opp` | goals the opponent conceded per match in its last 10 matches |
| `rest_days` | days since the team's previous World Cup match |
| `is_host` | 1 if the team is a host (Mexico, Canada, United States) |
| `is_knockout` | 1 for Round of 32 onwards |
| `goals_scored` | **target**: goals the team scored (extra time included, shoot-outs excluded) |

In [7]:
rows = []
for match_id, m in enumerate(matches, start=1):
    sides = {m["team1"]: ("team1", "team2", m["goals1"]), m["team2"]: ("team2", "team1", m["goals2"])}
    for team in sorted(sides):
        me, other, goals = sides[team]
        opp = m[other]
        rows.append({
            "row_id": len(rows) + 1,
            "match_id": match_id,
            "date": m["date"],
            "team": team,
            "opp": opp,
            "fifa_points_team": fifa_points[team],
            "squad_value_team_eur_m": squad_value[team],
            "squad_value_opp_eur_m": squad_value[opp],
            "form_gf_pg_team": round(goals_for_per_game(team, KICKOFF), 1),
            "form_ga_pg_opp": round(goals_against_per_game(opp, KICKOFF), 1),
            "rest_days": m["rest_" + me],
            "is_host": int(team in HOSTS),
            "is_knockout": m["is_knockout"],
            "goals_scored": goals,
        })

df = pd.DataFrame(rows)
df.to_csv("wc2026_2_2_dataset.csv", index=False)
df.head(10)

,row_id,match_id,date,team,opp,fifa_points_team,squad_value_team_eur_m,squad_value_opp_eur_m,form_gf_pg_team,form_ga_pg_opp,rest_days,is_host,is_knockout,goals_scored
0,1,1,2026-06-11,Mexico,South Africa,1687.48,191.85,49.25,1.6,1.0,7,1,0,2
1,2,1,2026-06-11,South Africa,Mexico,1428.38,49.25,191.85,1.5,0.4,7,0,0,0
2,3,2,2026-06-11,Czechia,Korea Republic,1505.74,188.18,139.05,2.2,1.2,7,0,0,1
3,4,2,2026-06-11,Korea Republic,Czechia,1591.63,139.05,188.18,1.5,0.9,7,0,0,2
4,5,3,2026-06-12,Bosnia and Herzegovina,Canada,1387.22,146.40,198.65,2.0,0.4,7,0,0,1
5,6,3,2026-06-12,Canada,Bosnia and Herzegovina,1559.48,198.65,146.40,1.1,1.0,7,1,0,1
6,7,4,2026-06-12,Paraguay,United States,1505.35,153.65,385.60,1.2,1.7,7,0,0,1
7,8,4,2026-06-12,United States,Paraguay,1671.23,385.60,153.65,1.8,1.0,7,1,0,4
8,9,5,2026-06-13,Qatar,Switzerland,1450.31,19.93,332.50,0.7,0.8,7,0,0,1
9,10,5,2026-06-13,Switzerland,Qatar,1650.06,332.50,19.93,2.2,1.5,7,0,0,1


## 8. Quick checks
Besides the size checks, this confirms the two rows of each match agree with each other,
and counts how many explanatory variables are shared with Task 2.1 (the limit is 4).

In [8]:
X_COLS = ["fifa_points_team", "squad_value_team_eur_m", "squad_value_opp_eur_m",
          "form_gf_pg_team", "form_ga_pg_opp", "rest_days", "is_host", "is_knockout"]
Y_COL = "goals_scored"

assert len(df) == 208                                   # 208 team-match rows
assert df["match_id"].value_counts().eq(2).all()        # exactly 2 rows per match
assert len(X_COLS) == 8                                 # 8 explanatory variables
assert df[X_COLS + [Y_COL]].isna().sum().sum() == 0     # nothing missing

# the opponent columns must equal the opponent's own row
pair = df.merge(df, left_on=["match_id", "opp"], right_on=["match_id", "team"], suffixes=("", "_o"))
assert (pair["squad_value_opp_eur_m"] == pair["squad_value_team_eur_m_o"]).all()

# variables used in Task 2.1, compared with the ones used here
X_COLS_2_1 = ["fifa_points_diff", "host_team1", "host_team2", "same_confederation",
              "round_number", "h2h_matches_played", "h2h_avg_goal_diff", "recent_form_diff"]
shared = sorted(set(X_COLS) & set(X_COLS_2_1))
print("Identical columns shared with 2.1:", shared if shared else "none")
print("Related ideas: FIFA points, host status, tournament stage, recent form (4, within the limit)")
print("All checks passed")
df[X_COLS + [Y_COL]].describe().round(2)

Identical columns shared with 2.1: none
Related ideas: FIFA points, host status, tournament stage, recent form (4, within the limit)
All checks passed


,fifa_points_team,squad_value_team_eur_m,squad_value_opp_eur_m,form_gf_pg_team,form_ga_pg_opp,rest_days,is_host,is_knockout,goals_scored
count,208.00,208.00,208.00,208.00,208.00,208.00,208.00,208.00,208.00
mean,1617.21,451.25,451.25,1.96,0.87,5.68,0.07,0.31,1.48
std,162.68,424.24,424.24,0.67,0.42,1.11,0.26,0.46,1.39
min,1275.58,19.93,19.93,0.70,0.20,3.00,0.00,0.00,0.00
25%,1505.35,139.05,139.05,1.30,0.50,5.00,0.00,0.00,0.00
50%,1602.12,332.50,332.50,1.90,0.90,6.00,0.00,0.00,1.00
75%,1745.07,589.90,589.90,2.50,1.00,7.00,0.00,1.00,2.00
max,1877.27,1520.00,1520.00,3.70,2.00,8.00,1.00,1.00,7.00


## 9. Download the dataset

In [9]:
try:
    from google.colab import files
    files.download("wc2026_2_2_dataset.csv")
except ImportError:
    print("Saved as wc2026_2_2_dataset.csv")

Saved as wc2026_2_2_dataset.csv
